# 01 — Scopus reproducibility

This notebook reproduces the **Scopus aggregation stage** of the Paper-1 methodology from **frozen platform exports**.

## What is and is not reproduced

| Included | Not included |
|----------|----------------|
| Loading `sources/scopus/q*/{A,B}/scopus.csv` | Live Scopus API / `pybliometrics` |
| Core RE/TE query selection (204 rows) | Re-running original web searches |
| Platform-level `Title` deduplication (164 rows) | API credentials |

Frozen exports are the **reproducibility boundary**. Historical exploratory live-search cells in the original notebook are archival only (`legacy/notebooks/scopus.ipynb`, ignored).


## Query selection

**Included (core funnel):** `q1`, `q2`, `q3`, `q4`, `q16`, `q17` (strategies A/B when present).

**Excluded from the core Paper-1 search:** Information Extraction and Generation families  
`q31–q34`, `q46–q49`, `q61–q64`.

Concatenation order is frozen to the historical discovery order that produced the legacy unique-title checkpoint (`legacy/checkpoints/scopus_reducido.csv`):  
`q1 → q17 → q2 → q16 → q4 → q3` (strategy **A** then **B**).


In [ ]:
from re_te_lowresources.scopus import (
    CORE_QUERY_ORDER,
    EXPECTED_CORE_ROWS,
    EXPECTED_UNIQUE_ROWS,
    default_repo_root,
    discover_scopus_exports,
    reproduce_scopus,
)

ROOT = default_repo_root()
print("Repository root:", ROOT)
print("Core query order:", CORE_QUERY_ORDER)


## Discover frozen exports


In [ ]:
exports = discover_scopus_exports(ROOT / "sources" / "scopus")
for path in exports:
    print(path.relative_to(ROOT))
print(f"\n{len(exports)} export files")


## Aggregate and deduplicate

Shared implementation: `re_te_lowresources.scopus.reproduce_scopus`.

- **Core concat** → `data/automatic/scopus/scopus_core.csv` (expected **204**)
- **Title unique, keep first** → `data/automatic/scopus/scopus_unique.csv` (expected **164**)

`scopus_core.csv` is the core Paper-1 search concat, not the historical all-query exploratory aggregate (446 rows).


In [ ]:
result = reproduce_scopus(ROOT, validate=True, write=True)

print(f"Scopus core records: {result.raw_rows} (expected {EXPECTED_CORE_ROWS})")
print(f"Scopus unique records: {result.dedup_rows} (expected {EXPECTED_UNIQUE_ROWS})")
print("Wrote:", result.core_path.relative_to(ROOT))
print("Wrote:", result.unique_path.relative_to(ROOT))

assert result.raw_rows == EXPECTED_CORE_ROWS
assert result.dedup_rows == EXPECTED_UNIQUE_ROWS
print("PASS")


## Compact checkpoint summary


In [ ]:
core = result.core
unique = result.unique

summary = {
    "core_rows": len(core),
    "unique_rows": len(unique),
    "columns": len(core.columns),
    "queries_in_core": sorted(core["query"].unique()),
    "rows_per_query_core": core["query"].value_counts().sort_index().to_dict(),
}
for key, value in summary.items():
    print(f"{key}: {value}")

# TERL still has two bibliographic rows before later selection-stage normalization.
terl = core[core["Title"].astype(str).str.contains("TERL:", case=False, na=False)]
print(f"TERL rows in core concat: {len(terl)}")
